## DragonHPC and cuPhoton - Bootstrap

Placeholder for the session on [DragonHPC](https://dragonhpc.github.io/dragon/doc/_build/html/index.html) and [cuPhoton](https://github.com/NVIDIA/cuPhoton). The cells below run one small computation with each: a `multiprocessing` pool on the Dragon runtime, and a cuPhoton dipole fit on synthetic stamps.

In [ ]:
import subprocess

import cuphoton

print("cuphoton", cuphoton.__version__)
print(subprocess.run(["dragon", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
%%writefile dragon_bootstrap.py
import dragon
from multiprocessing import Pool, set_start_method


def f(x):
    return x * x


if __name__ == "__main__":
    set_start_method("dragon")
    with Pool(2) as p:
        print(p.map(f, [1, 2, 3]))

In [ ]:
run = subprocess.run(["dragon", "dragon_bootstrap.py"], capture_output=True, text=True)
print(run.stdout, run.stderr, sep="")
run.check_returncode()

In [ ]:
import numpy as np
from cuphoton.xfit import GaussianDipoleModel, fit_dipoles

# The three stamps of cuPhoton's quickstart: render them from known parameters, perturb, fit on the GPU.
model = GaussianDipoleModel((17, 21), dtype=np.float64)
true_parameters = np.asarray(
    [
        [7.0, 1.7, 1.3, 0.20, -2.3, -1.8, 2.5, 1.4],
        [5.5, 1.4, 1.8, -0.15, -3.1, 1.6, 2.2, -1.4],
        [8.0, 1.9, 1.5, 0.35, -1.8, -2.0, 3.2, 2.1],
    ]
)
images = np.asarray(model.evaluate(true_parameters, mode="difference"), dtype=np.float64)
initial = true_parameters.copy()
initial[:, 0] *= 0.9
initial[:, 4:8] += np.asarray([0.25, -0.20, -0.20, 0.25])

result = fit_dipoles(images, model=model, initial=initial, backend="cupy")
print(result.backend, "on", result.device, "| converged", int(np.count_nonzero(result.converged)), "of", len(images),
      "| largest parameter error", float(np.abs(result.parameters - true_parameters).max()))